# 02 — Retrieval embedder (10-K QA)

Fine-tunes `BAAI/bge-base-en-v1.5` on question→context pairs drawn from real
10-K filings.

| | |
|---|---|
| Base | `BAAI/bge-base-en-v1.5` (`bge-large` if VRAM allows) |
| Data | `virattt/financial-qa-10K` |
| Loss | `CachedMultipleNegativesRankingLoss` (GradCache) |
| T4 | effective batch 64, `mini_batch_size` 8–16, 1 epoch, lr 2e-5 |

**Train on 10-K QA, not FiQA.** This is the single most important choice here and
it is evidence-based: fine-tuning `bge-large` on FiQA in a prior project improved
FiQA NDCG@10 by 2.3% and *cost* 11.6% Hit@1 on filing retrieval. FiQA is
retail-investor forum discussion; filings are SEC prose. We train in-domain and
evaluate on **both**, so the trade-off is visible instead of hidden behind the
one favourable number.

Things that decide whether this works:
- **Batch size is the in-batch negative count** for this loss — it matters more
  than epochs, and GradCache is what lets a 16GB T4 hold an effective 64.
- `BatchSamplers.NO_DUPLICATES`: two rows sharing a positive in one batch trains
  the model against a true positive.
- The BGE query-instruction prefix is used in **neither** training nor inference.
  A mismatch is worse than skipping it; `configs/default.yaml` mirrors this.
- FinanceBench overlap is checked and dropped before training, and the count is
  printed even when it is zero.

In [ ]:
# Confirm we actually have the T4 this recipe is written for.
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

import torch
assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > T4 GPU."
print(f"torch {torch.__version__} | {torch.cuda.get_device_name(0)} | "
      f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Checkpoints MUST live somewhere that survives the VM (section 5.5).
# /content is ephemeral - it vanishes with the runtime, which is exactly the
# failure checkpointing exists to defend against.
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/affa'
os.makedirs(DRIVE_ROOT, exist_ok=True)
print('checkpoints ->', DRIVE_ROOT)

In [ ]:
# Clone or update the repo, and verify it is current. Re-running this notebook
# from the top after a disconnect must not silently train an old revision.
import os, subprocess

REPO_URL = 'https://github.com/abhinaba01/agentic-financial-filing-analysis.git'
REPO_DIR = '/content/agentic-financial-filing-analysis'

if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', '--all'], check=True)

local  = subprocess.run(['git', '-C', REPO_DIR, 'rev-parse', 'HEAD'],
                        capture_output=True, text=True).stdout.strip()
remote = subprocess.run(['git', '-C', REPO_DIR, 'rev-parse', '@{u}'],
                        capture_output=True, text=True).stdout.strip()

if remote and local != remote:
    print(f'repo is BEHIND origin (local {local[:8]} != remote {remote[:8]})')
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)
    print('pulled; RESTART THE RUNTIME so the new code is imported')
else:
    print(f'repo is current at {local[:8]}')

os.chdir(REPO_DIR)

In [ ]:
# datasets<4.0 is REQUIRED, not a preference: finer-139, financial_phrasebank
# and finqa are loading-script datasets, and datasets>=4.0 removed script
# execution entirely. The parquet mirrors are NOT equivalent - at least one is
# deduplicated, which changes the splits and breaks comparability.
%pip install -q -e ".[train,eval]"
%pip install -q "datasets>=2.19,<4.0"

import datasets, transformers
print('datasets', datasets.__version__, '| transformers', transformers.__version__)
assert int(datasets.__version__.split('.')[0]) < 4, (
    'datasets>=4.0 cannot execute loading scripts; pin datasets>=2.19,<4.0'
)

In [ ]:
SEED            = 42
TRAIN_SAMPLES   = None    # the dataset is small enough to use in full
EVAL_SAMPLES    = 500
MAX_LENGTH      = 384
BATCH_SIZE      = 64      # effective batch = in-batch negative count
MINI_BATCH_SIZE = 8       # what actually sits on the T4 at once (GradCache)
EPOCHS          = 1
LR              = 2e-5
SAVE_STEPS      = 200

CKPT_DIR = f'{DRIVE_ROOT}/retrieval_embedder'
print(CKPT_DIR)

## Checkpointing and resume

Colab runtimes disconnect, get recycled, and hit idle timeouts. Everything below
is built so a crash costs minutes, not the whole run.

**What resume restores:** model weights, optimizer moments, LR-scheduler
position, RNG state, global step, and dataloader position. That is why we resume
rather than "just train again from the saved weights" — restarting the optimizer
and the LR schedule from scratch is a *different run*, and its loss curve will
not join up with the first half.

**The cell below is idempotent.** Re-run it after a crash and it resumes
automatically, with no code edit.

**Determinism is a precondition.** `SEED`, `TRAIN_SAMPLES` and `EVAL_SAMPLES` are
written into the checkpoint directory as JSON, and the resume path *refuses* to
continue if they no longer match. Changing any of them after a crash means the
global step now points into different data and the resumed run is silently
meaningless (anti-pattern #14).

**Disk:** a full checkpoint is roughly 3–4× model size — fp32 weights plus two
AdamW moments — so `save_total_limit=2` is required, not tidiness, against
Drive's 15GB free tier. `save_steps` is set for ~15–20 minutes of training, not
per epoch: an epoch here is 40+ minutes and a disconnect at minute 39 loses all
of it.

In [ ]:
# SentenceTransformerTrainer subclasses HF Trainer, so the same resume applies.
!python training/train_retrieval.py \
    --output-dir "{CKPT_DIR}" \
    --seed {SEED} \
    --eval-samples {EVAL_SAMPLES} \
    --max-length {MAX_LENGTH} \
    --batch-size {BATCH_SIZE} \
    --mini-batch-size {MINI_BATCH_SIZE} \
    --epochs {EPOCHS} \
    --learning-rate {LR} \
    --save-steps {SAVE_STEPS}

## Test the resume path — do not assume it

Untested resume logic is usually broken resume logic, and the moment you find
out is the moment you have already lost the run.

1. Run the training cell above and let it write at least two checkpoints.
2. **Runtime → Interrupt execution** (or just let the runtime die).
3. Re-run the training cell *unchanged*.

What you should see: `resuming from .../checkpoint-N`, and the loss continuing
from where it stopped rather than restarting near its initial value. If step
numbering restarts at 0, resume is not working — fix that before starting the
real run.

## Evaluate on both corpora

Retrieval evaluation makes **no LLM calls**, so before/after comparisons are
free — run them often.

Report both tables. If in-domain training helped filings and hurt FiQA, that is
the expected result and the interesting one; reporting only the corpus that
improved would be exactly the kind of selective reporting this project exists to
avoid.

In [ ]:
# FiQA: forum discussion. Expect this to move less, or to regress.
!affa-eval retrieval \
    --test-set BeIR/fiqa \
    --model "{CKPT_DIR}/final" \
    --baseline BAAI/bge-base-en-v1.5 \
    --limit 500 \
    --output eval_results/retrieval_fiqa.json

In [ ]:
# FinanceBench: SEC filing prose. This is the corpus the product actually serves.
!affa-eval retrieval \
    --test-set PatronusAI/financebench \
    --model "{CKPT_DIR}/final" \
    --baseline BAAI/bge-base-en-v1.5 \
    --output eval_results/retrieval_financebench.json

In [ ]:
# Push the model and a card carrying the REAL numbers and the subset size.
# A model card with aspirational numbers is worse than no card.
from huggingface_hub import notebook_login
notebook_login()

HUB_ID = 'YOUR_USERNAME/affa-retrieval-embedder'

card = """---
license: apache-2.0
tags: [finance, sec-filings, affa]
---

# affa-retrieval-embedder

Fine-tuned for the Agentic Financial Filing Analyst.

Bi-encoder fine-tuned on 10-K question/context pairs for filing retrieval.

## Measured results

Fill these in from the evaluation cell above. Report the **test** split score,
the **baseline measured on the same data with the same protocol**, and the
training subset size. Do not paste a number from a paper here.

| metric | this model | baseline | notes |
|---|---:|---:|---|
| (fill in) | | | |

- Training subset: `TRAIN_SAMPLES` (state the number actually used)
- Seed: `SEED`
- Checkpoint selected on: validation split
- Test split touched: once

## Not financial advice

Research and educational use only.
"""

import pathlib
pathlib.Path(f'{CKPT_DIR}/final/README.md').write_text(card, encoding='utf-8')
print('model card written; review it before pushing')